# Portfolio-opdracht 2: Voorspel verhuuraantallen (CityBike)

**Deel: Opdracht 2 — Time series feature engineering** · Ahmed Bashabeeb

Dit notebook is het tweede deel van ons portfolio. Aan het eind worden de delen samengevoegd in
`Portfolio2_CityBike.ipynb`. De data (`train.csv`, `test.csv`) staat in de map `data/`. Het
inladen en opschonen van de data gebeurt met de code uit Opdracht 1 (`opdracht1_eda.ipynb`).

**Inhoud**

- Voorbereiding: imports, instellingen en data uit Opdracht 1
- 2.1 Evaluatiemethode · 2.2 Stationariteit (Dickey-Fuller) · 2.3 Trend
- 2.4 Seizoenspatronen met Fourier-analyse · 2.5 Cycli, autocorrelatie en lags
- 2.6 Overige tijdgebaseerde features · 2.7 Klasse `FeatureEngineer`
- 2.8 Invloed van de features · 2.9 Gekozen features

## Voorbereiding: imports, instellingen en data uit Opdracht 1

We gebruiken de klasse `DataLoader` (paragraaf 1.2) en de functie `add_calendar_features`
(paragraaf 1.5) uit Opdracht 1, en schonen de data op zoals in paragraaf 1.9: `hum == 0`
vervangen door interpolatie en 29–30 oktober 2012 verwijderen.

In [ ]:
# Bron: Week 10 – Regression Time Series 2 (import-cel)
# Bron: Week 11 – Time series analysis 3 (imports statsmodels)

# Importeren van de packages die nodig zijn voor de feature engineering
import os
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.fft import fft
from scipy.signal import find_peaks
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.deterministic import (CalendarFourier,
                                           DeterministicProcess)
from statsmodels.tsa.stattools import adfuller
from xgboost import XGBRegressor

# Waarschuwingen onderdrukken voor een overzichtelijk notebook
warnings.filterwarnings('ignore')

# Vaste instellingen
DATA_FOLDER = os.path.join('..', 'data')     # map met de Kaggle-bestanden
RANDOM_STATE = 42                            # reproduceerbare resultaten
HORIZON = 19 * 24                            # 456 uur, zoals de Kaggle-test
HOLDOUT_START = pd.Timestamp('2012-11-12')   # laatste 19 dagen = testset
pd.set_option('display.max_colwidth', 150)
plt.rcParams['figure.figsize'] = (12, 4)
sns.set_style('whitegrid')

In [ ]:
# Bron: Opdracht 1, paragraaf 1.2 (klasse DataLoader), 1.5
#       (add_calendar_features) en 1.9 (opschonen); opdracht1_eda.ipynb

class DataLoader:
    """Leest de bestanden van de Kaggle-competitie in en schoont ze op."""

    def __init__(self, folder='.'):
        self.folder = folder

    def load(self, file_name):
        """Lees een CSV-bestand in met 'date_hour' als datetime."""
        path = os.path.join(self.folder, file_name)
        return pd.read_csv(path, parse_dates=['date_hour'])

    @staticmethod
    def clean(data, outlier_days):
        """Schoon de data op (onderbouwing: paragraaf 1.4 en 1.9).

        - sorteren op tijd en dubbele uren verwijderen;
        - luchtvochtigheid 0 (meetfout) vervangen door interpolatie;
        - uitzonderlijke dagen (outliers) verwijderen.
        """
        data = data.sort_values('date_hour')
        data = data.drop_duplicates('date_hour').copy()
        data.loc[data['hum'] == 0, 'hum'] = np.nan
        data['hum'] = data['hum'].interpolate()
        days = data['date_hour'].dt.normalize()
        data = data[~days.isin(pd.to_datetime(outlier_days))]
        return data.reset_index(drop=True)


def add_calendar_features(data):
    """Voeg tijdselementen toe die uit 'date_hour' volgen."""
    data = data.copy()
    time = data['date_hour']
    data['hour'] = time.dt.hour
    data['weekday'] = time.dt.dayofweek           # 0 = maandag, 6 = zondag
    data['day_name'] = time.dt.day_name()
    data['week'] = time.dt.isocalendar().week.astype(int)
    data['month'] = time.dt.month
    data['year'] = time.dt.year - 2011            # 0 = 2011, 1 = 2012
    data['dayofyear'] = time.dt.dayofyear
    # werkdag = maandag t/m vrijdag en geen feestdag
    is_weekday = data['weekday'] < 5
    data['workingday'] = (is_weekday & (data['holiday'] == 0)).astype(int)
    return data


# Data inladen en opschonen zoals in Opdracht 1 (paragraaf 1.9)
OUTLIER_DAYS = ['2012-10-29', '2012-10-30']
loader = DataLoader(DATA_FOLDER)
train = loader.clean(loader.load('train.csv'), OUTLIER_DAYS)
test = loader.clean(loader.load('test.csv'), outlier_days=[])
print(f'train: {len(train)} rijen | test: {len(test)} rijen')

# 2. Opdracht 2: Time series feature engineering

In deze opdracht onderzoeken we stationariteit, trend, seizoenspatronen en cycli, maken we daarvan
features en meten we wat hun invloed is op de voorspellingen. Dit was een **iteratief proces**:
features toevoegen → model testen → nieuwe features proberen (2.8).

## 2.1 Evaluatiemethode

Om de invloed van features en modellen eerlijk te meten, leggen we eerst de evaluatie vast.

- **Niet husselen.** Bij tijdreeksen mag je de data niet willekeurig verdelen: dan traint het model
  op de toekomst en wordt de score te optimistisch (*data leakage*). Cross-validatie bij tijdreeksen
  werkt daarom anders dan gewone k-fold (notebook *Cross validation*; De Haagse Hogeschool,
  z.d.-a): elke fold traint op het **verleden** en valideert op de periode **erna**
  (`TimeSeriesSplit`; opdracht 11 van *Regression Time Series 2*, De Haagse Hogeschool, z.d.-d;
  *Week 11 application remarks*, De Haagse Hogeschool, z.d.-j; Holdgraf, z.d.).
- **Trainset en testset.** Net als in opdracht 3.1 van *Regression Time Series 1* en de klasse
  `RegressionModels` uit *Regression Time Series 3* (`train_test_split(shuffle=False)`; De Haagse
  Hogeschool, z.d.-c, z.d.-e) houden we de **laatste periode**, 19 dagen (12–30 november 2012),
  apart als **testset**. Die gebruiken we pas aan het eind, één keer per model. De hyperparameters
  kiezen we met cross-validatie op de trainset. Daarna trainen we het beste model opnieuw op álle
  data voor Kaggle (Radhakrishnan, 2023a).
- **Folds op de trainset.** Elke validatieperiode is **456 uur (19 dagen)**, net zo lang als de
  Kaggle-test:
    - 4 **herfst-folds** (`TimeSeriesSplit(n_splits=4, test_size=456)`): de laatste 2,5 maand vóór de
      testset;
    - 3 **winter-folds**: 1–19 december 2011, 9–27 januari 2012 en 6–24 februari 2012.

  Meer folds geven een betrouwbaarder gemiddelde, maar kosten meer rekentijd; met 7 folds zitten
  beide seizoenen er goed in.
- **Waarom winter-folds? (iteratie na Kaggle-feedback)** In onze eerste versie gebruikten we alleen
  herfst-folds. Op Kaggle bleek dat modellen op log-schaal (lineaire regressie, SARIMAX) veel beter
  scoorden dan onze validatie voorspelde (zie 4.3). December heeft lage aantallen en gedraagt zich
  anders dan de drukke herfst. Daarom valideren we nu ook op winterperiodes. De code voor de
  winter-folds komt niet in de lesstof voor (Claude, 2026. Prompt 2: Winter-folds voor CV).

**Metrics** (opdracht 2 van *Regression 1*; De Haagse Hogeschool, z.d.-b): $R^2$ (verklaarde
variantie), MAE (gemiddelde absolute fout),
MSE (gemiddelde gekwadrateerde fout) en **RMSE** $=\sqrt{\text{MSE}}$, de maatstaf van Kaggle:
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$
De RMSE heeft dezelfde eenheid als `cnt` (fietsen per uur) en straft grote fouten zwaar af.

In [ ]:
# Bron: Week 10 – Regression Time Series 2, opdracht 11 (TimeSeriesSplit)
# Bron: Week 11 – application remarks (TimeSeriesSplit)
# Bron: Week 8 – Regression 1, opdracht 2 (R2, MAE, MSE en RMSE)
# Bron: Week 8 – Cross validation (cross_val_score)
# Bron: Week 9 – Regression Time Series 1, opdracht 3.1 (testset)
# Prompt 2: Winter-folds voor CV

WINTER_STARTS = ['2011-12-01', '2012-01-09', '2012-02-06']
N_AUTUMN = 4


def make_folds(data, n_autumn=N_AUTUMN, winter_starts=WINTER_STARTS):
    """Maak de folds: (trainingsrijen, validatierijen) per fold.

    Elke fold traint op alle rijen vóór de validatieperiode van 456 uur.
    """
    autumn = TimeSeriesSplit(n_splits=n_autumn, test_size=HORIZON)
    folds = list(autumn.split(data))
    for start in pd.to_datetime(winter_starts):
        end = start + pd.Timedelta(hours=HORIZON)
        train_idx = np.where(data['date_hour'] < start)[0]
        val_idx = np.where((data['date_hour'] >= start)
                           & (data['date_hour'] < end))[0]
        folds.append((train_idx, val_idx))
    return folds


def rmse(y_true, y_pred):
    """Root Mean Squared Error (de maatstaf van Kaggle)."""
    return np.sqrt(mean_squared_error(y_true, y_pred))


def regression_metrics(y_true, y_pred):
    """R2, MAE, MSE en RMSE (zoals in de les Regression 1)."""
    mse = mean_squared_error(y_true, y_pred)
    return {'R2': r2_score(y_true, y_pred),
            'MAE': mean_absolute_error(y_true, y_pred),
            'MSE': mse, 'RMSE': np.sqrt(mse)}


# Trainset en testset: de laatste 19 dagen zijn de testset (holdout)
train_part = train[train['date_hour'] < HOLDOUT_START].reset_index(drop=True)
holdout = train[train['date_hour'] >= HOLDOUT_START].reset_index(drop=True)
CV = make_folds(train_part)


def cv_rmse(model, X, y):
    """Gemiddelde RMSE over alle folds van de trainset."""
    scores = cross_val_score(model, X, y, cv=CV,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean()


fold_overview = []
for fold, (train_idx, val_idx) in enumerate(CV, start=1):
    fold_overview.append({
        'fold': fold,
        'type': 'herfst' if fold <= N_AUTUMN else 'winter',
        'training t/m': train_part['date_hour'][train_idx].max(),
        'validatie van': train_part['date_hour'][val_idx].min(),
        'validatie t/m': train_part['date_hour'][val_idx].max()})
print(f'trainset: {len(train_part)} uur | testset: '
      f'{holdout["date_hour"].min():%d-%m-%Y} t/m '
      f'{holdout["date_hour"].max():%d-%m-%Y} ({len(holdout)} uur)')
pd.DataFrame(fold_overview)

## 2.2 Stationariteit (Augmented Dickey-Fuller-test)

Een tijdreeks is **stationair** als het gemiddelde, de variantie en de autocorrelatie in de tijd
constant zijn (Radhakrishnan, 2023d). Modellen als ARMA vereisen dat. We testen dit met de
**Augmented Dickey-Fuller-test**, zoals in opdracht 5 van *Time series analysis 3* (De Haagse
Hogeschool, z.d.-h):

- nulhypothese $H_0$: de reeks is **niet** stationair;
- p-waarde < 0,05 → $H_0$ verwerpen → de reeks is stationair.

Een niet-stationaire reeks kun je stationair maken door te **differentiëren**: $y_t - y_{t-1}$. Dat is
de *I* (integrated) in ARIMA.

In [ ]:
# Bron: Week 11 – Time series analysis 3, opdracht 5 (adfuller)
# Bron: Week 11 – slides Timeseries analyse 3 (ADF, differentiëren)

# Regelmatige reeksen: per uur (ontbrekende uren geïnterpoleerd) en per dag
hourly = train.set_index('date_hour')['cnt'].asfreq('h').interpolate()
daily = (train.set_index('date_hour')['cnt'].resample('D').mean()
         .asfreq('D').interpolate())


def adf_test(series, name):
    """Augmented Dickey-Fuller-test (H0: de reeks is niet stationair)."""
    statistic, p_value = adfuller(series.dropna(), autolag='AIC')[:2]
    return {'reeks': name, 'ADF-statistiek': round(statistic, 2),
            'p-waarde': round(p_value, 4),
            'stationair (p < 0,05)': p_value < 0.05}


pd.DataFrame([adf_test(hourly, 'cnt per uur'),
              adf_test(daily, 'cnt per dag (gemiddelde)'),
              adf_test(daily.diff(), 'cnt per dag, gedifferentieerd')])

**Toelichting.**

- **Per uur** is de reeks volgens de ADF-test stationair (p < 0,0001): het sterke dagritme trekt de
  reeks steeds terug naar hetzelfde niveau.
- **Per dag** is de reeks **niet** stationair (p = 0,29): het gemiddelde verandert door de trend en
  het jaarseizoen.
- **Na differentiëren** is de dagreeks wel stationair (p < 0,0001).
- Gevolgen: (1) een ARIMA-model op dagniveau zou $d = 1$ nodig hebben; (2) voor SARIMAX op uurniveau
  over een korte periode (8 weken) nemen we $d = 0$ (3.8); (3) de machine-learning modellen krijgen de
  trend als feature (2.3).

## 2.3 Trend

Een **trend** is een langzame verandering van het niveau over een lange periode. We onderzoeken
de trend op twee manieren uit de lessen:

1. **Voortschrijdende gemiddelden** (*moving averages*) met verschillende window-groottes. Een
   window zo groot als een seizoensperiode (7 dagen, 365 dagen) middelt dat seizoen weg
   (Radhakrishnan, 2023b; Kaggle-cursus les 2, Holbrook, z.d.-f).
2. **`DeterministicProcess`** (Kaggle-cursus les 1 en 2, Holbrook, z.d.-c, z.d.-f; opdracht 5–7
   van *Regression Time Series 2*, De Haagse Hogeschool, z.d.-d; *Week 10 application remarks*,
   De Haagse Hogeschool, z.d.-i): een tijd-dummy $t$ (en $t^2$, $t^3$ bij een hogere *order*) als
   feature voor lineaire regressie, $y(t) = a + b\,t + c\,t^2 + \dots$ Omdat het een functie van de
   tijd is, kun je hiermee **extrapoleren** naar de toekomst.

In [ ]:
# Bron: Kaggle Time Series, les 2 – Trend (Holbrook)
# Bron: Week 9 – Time series analysis 1, opdracht 2 en 3

fig, ax = plt.subplots()
ax.plot(daily, alpha=0.3, color='grey', label='gemiddelde per dag')
for window in [7, 30, 365]:
    moving_average = daily.rolling(window, center=True,
                                   min_periods=window // 2).mean()
    ax.plot(moving_average, label=f'voortschrijdend gemiddelde ({window} d)')
ax.set(title='Trend met voortschrijdende gemiddelden', ylabel='cnt per uur')
ax.legend()
plt.show()

# Groei: vergelijk dezelfde maanden (januari t/m november)
same_months = add_calendar_features(train).query('month <= 11')
mean_per_year = same_months.groupby('year')['cnt'].mean()
growth = (mean_per_year[1] / mean_per_year[0] - 1) * 100
print(f'Gemiddelde jan-nov 2011: {mean_per_year[0]:.0f} | '
      f'2012: {mean_per_year[1]:.0f} | groei: +{growth:.0f}%')

Welke **order** van de trend past het best? Net als in opdracht 5–7 van *Regression Time Series 2*
(De Haagse Hogeschool, z.d.-d) fitten we lineaire regressie met `DeterministicProcess` voor
order 1, 2 en 3, hier op het gemiddelde per dag en samen met Fourier-termen voor het jaarseizoen
(zie 2.4; Kaggle-cursus les 3, Holbrook, z.d.-d). We trainen op de trainset en extrapoleren met
`out_of_sample` naar de testset en de Kaggle-periode (t/m 19 december), zoals in Kaggle-cursus
les 2 (Holbrook, z.d.-f).

In [ ]:
# Bron: Week 10 – Regression Time Series 2, opdracht 5, 6 en 7
# Bron: Kaggle Time Series, les 2 – Trend (Holbrook)
# Bron: Kaggle Time Series, les 3 – Seasonality (Holbrook)

daily_train = daily[daily.index < HOLDOUT_START]
daily_test = daily[daily.index >= HOLDOUT_START]
future_days = pd.date_range(HOLDOUT_START, '2012-12-19', freq='D')

trend_rows = []
fig, ax = plt.subplots()
ax.plot(daily, color='grey', alpha=0.4, label='werkelijk (gemiddelde per dag)')
for order in [1, 2, 3]:
    dp = DeterministicProcess(index=daily_train.index, constant=False,
                              order=order, drop=True,
                              additional_terms=[CalendarFourier('YE', 3)])
    X_in = dp.in_sample()
    X_out = dp.out_of_sample(steps=len(future_days),
                             forecast_index=future_days)
    trend_model = Pipeline([('scale', StandardScaler()),
                            ('lr', LinearRegression())])
    trend_model.fit(X_in, daily_train)
    fitted = pd.Series(trend_model.predict(X_in), index=X_in.index)
    forecast = pd.Series(trend_model.predict(X_out), index=future_days)
    trend_rows.append({
        'order': order,
        'R2 trainset': r2_score(daily_train, fitted),
        'RMSE testset (per dag)': rmse(daily_test,
                                       forecast.loc[daily_test.index]),
        'voorspelling 19 dec': forecast.iloc[-1]})
    ax.plot(pd.concat([fitted, forecast]), label=f'order {order}')
ax.axvline(HOLDOUT_START, color='black', linestyle='--', label='start testset')
ax.set(title='Trend (DeterministicProcess) + jaarseizoen, met extrapolatie',
       ylabel='cnt per uur')
ax.legend()
plt.show()
pd.DataFrame(trend_rows).round(2)

**Conclusie trend.**

- De voortschrijdende gemiddelden laten een duidelijke **stijgende trend** zien: in 2012 gemiddeld
  **+65%** verhuur ten opzichte van dezelfde maanden in 2011. Het 7-daagse gemiddelde volgt nog het
  weer, het 30-daagse het jaarseizoen; het 365-daagse gemiddelde stijgt geleidelijk.
- Alle drie de ordes passen even goed op de trainset ($R^2 \approx 0{,}78$), maar **order 1**
  voorspelt de testset het best (RMSE 59 per dag, tegen 63–66). Hogere ordes geven een gebogen
  trend die bij extrapolatie sneller de verkeerde kant op gaat. We kiezen dus **order 1**: de kolom
  `trend` uit `DeterministicProcess`.
- De Fourier-termen voor het jaarseizoen zorgen ervoor dat de lijn in de winter daalt. Een trend
  alleen kan dat niet (2.4).
- **Bomen kunnen een trend niet extrapoleren** (*Week 10 application remarks*, De Haagse Hogeschool,
  z.d.-i; Kaggle-cursus les 5, Holbrook, z.d.-b). Voor bomen gebruiken we daarom
  `year` (niveau per jaar); voor lineaire regressie en het hybride model de doorlopende `trend`.